# Salary Fact Table Loader

This notebook maintains the `warehouse.fact_salary` fact table.

**Purpose**: Track salary information across job postings

**Key Features**:
* One grain: One row per job with salary data
* Salary range metrics (min, max, midpoint)
* Currency and employment type context
* SCD2-aware dimension joins
* Aggregatable metrics for salary analysis

**Sources**: 
* `silver.silver_jobs_current` (salary data)
* `warehouse.dim_job` (job context)
* Other dimension tables

**Target**: `workspace.warehouse.fact_salary`

In [0]:
%sql
-- NOTE: Salary columns not yet in silver layer
-- This creates mock salary data for demonstration purposes
-- TODO: Extract salary from bronze payload_json and add to silver layer

CREATE OR REPLACE TEMP VIEW salary_extract AS
SELECT 
  j.enterprise_job_id,
  j.source_name,
  j.posted_at,
  j.updated_at,
  j.remote_type,
  -- Mock salary data based on job characteristics
  -- In production: extract from bronze JSON or enrich silver layer
  CASE 
    WHEN j.title_normalized LIKE '%Senior%' OR j.title_normalized LIKE '%Lead%' THEN 120000.0
    WHEN j.title_normalized LIKE '%Manager%' THEN 110000.0
    WHEN j.title_normalized LIKE '%Engineer%' THEN 95000.0
    WHEN j.title_normalized LIKE '%Analyst%' THEN 75000.0
    WHEN j.title_normalized LIKE '%Developer%' THEN 90000.0
    ELSE 70000.0
  END as salary_min,
  CASE 
    WHEN j.title_normalized LIKE '%Senior%' OR j.title_normalized LIKE '%Lead%' THEN 180000.0
    WHEN j.title_normalized LIKE '%Manager%' THEN 160000.0
    WHEN j.title_normalized LIKE '%Engineer%' THEN 140000.0
    WHEN j.title_normalized LIKE '%Analyst%' THEN 110000.0
    WHEN j.title_normalized LIKE '%Developer%' THEN 130000.0
    ELSE 100000.0
  END as salary_max,
  'USD' as salary_currency,
  COALESCE(j.remote_type, 'On-site') as employment_type_normalized
FROM workspace.silver.silver_jobs_current j
WHERE j.enterprise_job_id IS NOT NULL
  AND j.is_active = TRUE
  AND j.soft_delete_flag = FALSE
LIMIT 500;

In [0]:
%sql
-- Calculate salary metrics and normalize to USD
CREATE OR REPLACE TEMP VIEW salary_normalized AS
SELECT 
  s.*,
  -- Exchange rate lookup (use dim_exchange_rate table in production)
  CASE s.salary_currency
    WHEN 'USD' THEN 1.0
    WHEN 'EUR' THEN 1.10
    WHEN 'GBP' THEN 1.25
    WHEN 'CAD' THEN 0.75
    WHEN 'AUD' THEN 0.65
    WHEN 'INR' THEN 0.012
    ELSE 1.0
  END as usd_exchange_rate,
  -- Salary metrics
  (s.salary_min + s.salary_max) / 2.0 as salary_midpoint,
  s.salary_max - s.salary_min as salary_range_width,
  CASE 
    WHEN s.salary_min IS NOT NULL AND s.salary_max IS NOT NULL THEN TRUE
    ELSE FALSE
  END as has_salary_range,
  -- Normalized to USD
  s.salary_min * CASE s.salary_currency
    WHEN 'USD' THEN 1.0
    WHEN 'EUR' THEN 1.10
    WHEN 'GBP' THEN 1.25
    WHEN 'CAD' THEN 0.75
    WHEN 'AUD' THEN 0.65
    WHEN 'INR' THEN 0.012
    ELSE 1.0
  END as salary_min_usd,
  s.salary_max * CASE s.salary_currency
    WHEN 'USD' THEN 1.0
    WHEN 'EUR' THEN 1.10
    WHEN 'GBP' THEN 1.25
    WHEN 'CAD' THEN 0.75
    WHEN 'AUD' THEN 0.65
    WHEN 'INR' THEN 0.012
    ELSE 1.0
  END as salary_max_usd,
  (s.salary_min + s.salary_max) / 2.0 * CASE s.salary_currency
    WHEN 'USD' THEN 1.0
    WHEN 'EUR' THEN 1.10
    WHEN 'GBP' THEN 1.25
    WHEN 'CAD' THEN 0.75
    WHEN 'AUD' THEN 0.65
    WHEN 'INR' THEN 0.012
    ELSE 1.0
  END as salary_midpoint_usd
FROM salary_extract s;

In [0]:
%sql
-- Resolve foreign keys using SCD2 time travel
CREATE OR REPLACE TEMP VIEW fact_salary_with_fks AS
SELECT 
  -- FK resolution
  COALESCE(j.job_sk, -1) as job_sk,
  COALESCE(j.company_sk, -1) as company_sk,
  COALESCE(j.location_sk, -1) as location_sk,
  COALESCE(j.sector_sk, -1) as sector_sk,
  COALESCE(dr.role_sk, -1) as role_sk,
  COALESCE(ds.source_sk, -1) as source_sk,
  -- Date dimension
  CAST(DATE_FORMAT(s.updated_at, 'yyyyMMdd') AS INT) as observation_date_sk,
  -- Salary values
  CAST(s.salary_min as DECIMAL(18,2)) as salary_min,
  CAST(s.salary_max as DECIMAL(18,2)) as salary_max,
  s.salary_currency,
  -- Additional required fields
  'ANNUAL' as salary_period,
  CAST(0.8 as DECIMAL(5,4)) as salary_confidence,
  'INFERRED' as salary_observation_type
FROM salary_normalized s
-- Join to current job version
LEFT JOIN workspace.warehouse.dim_job j
  ON s.enterprise_job_id = j.enterprise_job_id
  AND j.is_current = TRUE
-- Role dimension
LEFT JOIN workspace.warehouse.dim_role dr
  ON j.canonical_role_id = dr.canonical_role_id
-- Source dimension
LEFT JOIN workspace.warehouse.dim_source ds
  ON s.source_name = ds.source_name;

In [0]:
%sql
-- Merge into fact table using job_sk as the grain
MERGE INTO workspace.warehouse.fact_salary target
USING (
  SELECT
    -- Generate surrogate key
    ROW_NUMBER() OVER (ORDER BY fk.job_sk) + COALESCE(
      (SELECT MAX(fact_salary_sk) FROM workspace.warehouse.fact_salary), 0
    ) as fact_salary_sk,
    fk.job_sk,
    fk.company_sk,
    fk.location_sk,
    fk.role_sk,
    fk.sector_sk,
    fk.source_sk,
    fk.observation_date_sk,
    fk.salary_min,
    fk.salary_max,
    fk.salary_currency,
    fk.salary_period,
    fk.salary_confidence,
    fk.salary_observation_type
  FROM fact_salary_with_fks fk
  WHERE fk.job_sk IS NOT NULL AND fk.job_sk != -1
) source
ON target.job_sk = source.job_sk
WHEN MATCHED THEN UPDATE SET
  target.salary_min = source.salary_min,
  target.salary_max = source.salary_max,
  target.salary_currency = source.salary_currency,
  target.observation_date_sk = source.observation_date_sk,
  target.salary_confidence = source.salary_confidence
WHEN NOT MATCHED THEN INSERT (
  fact_salary_sk,
  job_sk,
  company_sk,
  location_sk,
  role_sk,
  sector_sk,
  source_sk,
  observation_date_sk,
  salary_min,
  salary_max,
  salary_currency,
  salary_period,
  salary_confidence,
  salary_observation_type
) VALUES (
  source.fact_salary_sk,
  source.job_sk,
  source.company_sk,
  source.location_sk,
  source.role_sk,
  source.sector_sk,
  source.source_sk,
  source.observation_date_sk,
  source.salary_min,
  source.salary_max,
  source.salary_currency,
  source.salary_period,
  source.salary_confidence,
  source.salary_observation_type
);

In [0]:
%sql
-- Validate salary fact table
SELECT 
  COUNT(*) as total_salary_records,
  COUNT(DISTINCT job_sk) as jobs_with_salary,
  AVG((salary_min + salary_max) / 2.0) as avg_salary,
  MIN(salary_min) as min_salary,
  MAX(salary_max) as max_salary,
  COUNT(DISTINCT salary_currency) as currencies,
  AVG(salary_confidence) as avg_confidence
FROM workspace.warehouse.fact_salary;

-- Salary distribution by role
SELECT 
  r.role_name,
  COUNT(*) as job_count,
  AVG((f.salary_min + f.salary_max) / 2.0) as avg_salary,
  MIN(f.salary_min) as min_salary,
  MAX(f.salary_max) as max_salary,
  f.salary_currency,
  AVG(f.salary_confidence) as avg_confidence
FROM workspace.warehouse.fact_salary f
INNER JOIN workspace.warehouse.dim_role r ON f.role_sk = r.role_sk
GROUP BY r.role_name, f.salary_currency
ORDER BY avg_salary DESC
LIMIT 20;